In [ ]:
import argparse
import random
import logging
import json
import pdb
import numpy as np
from src.apex import APEX_naive
from src.pagerank import PR
from src.path import DBPEDIA_DATA_DIR, YAGO_DATA_DIR, METAQA_DATA_DIR, METAQA_DATA_DIR_ABLATION, ENCODING
import os
from itertools import chain


#logging.basicConfig(format='[%(asctime)s] - %(message)s',
   #                 level=logging.DEBUG,
    #                filename = 'log.log', filemode= 'a')

from time import time

from src.base import YAGO, DBPedia, Freebase, MetaQA
from src.user import query_log_by_mids, query_log_by_topics
from src.glimpse import SummaryMethod, GLIMPSE
from src.apex import APEX_naive, APEX, DETAILED_LOGGING
from src.metrics import total_query_log_metrics, average_query_log_metrics

: 

In [ ]:
KG_MAPPING = {
    'YAGO': YAGO(rdf_gz='yagoFacts.gz', query_dir='final/', mid_dir='by-mid/'),
   'Freebase': Freebase(query_dir='queries/final/'),
    'DBPedia': '../PKG_EXP/DBpedia/dbpedia_files', #DBPedia(),
    'MetaQA': MetaQA(ablation=0),
}

METHODS = { 
    'glimpse': SummaryMethod(GLIMPSE, 'GLIMPSE'),
    'glimpse-2': SummaryMethod(GLIMPSE, 'GLIMPSE-2', power=2),
    'apex': SummaryMethod(APEX, 'APEX'),
    'apex-n': SummaryMethod(APEX_naive, 'APEX-N'),
    'pagerank': SummaryMethod(PR, 'PageRank')
}

: 

In [ ]:
dbpedia_dir = '../PKG_EXP/DBpedia/dbpedia_files'
#for i, file_name in enumerate(sorted(os.listdir(dbpedia_dir))):
graph = '/Users/au735195/REDI/BIKE_RENTAL/APEX_code/PKG_EXP/DBpedia/facts.gz'#os.path.join(dbpedia_dir, file_name)           
#print(graph)
KG = DBPedia(rdf_gz=graph)
K = int(0.0001 * KG.number_of_triples())

#KG.load()

print((KG.__dict__.values()))
print((KG.__dict__['triples_']))

#summary_methods = [METHODS[name] for name in args.method]

: 

In [ ]:
import networkx as nx

def parse_triples_to_graph_dictionary(triples_snapshots):
    """
    Convert KG triples snapshots into a dictionary of directed graphs.

    :param triples_snapshots: list of snapshots, where each snapshot is a dictionary of triples
    :return: dict, dictionary of directed graphs for each timestep
    """
    dynamic_graph = {}

    for t, snapshot in enumerate(triples_snapshots):
        # Create a directed graph for the current snapshot
        directed_graph = nx.DiGraph()

        # Parse the snapshot
        for subject, predicates in snapshot.items():
            for predicate, objects in predicates.items():
                for obj in objects:
                    # Add edge (subject -> object) with the predicate as an attribute
                    directed_graph.add_edge(subject, obj, predicate=predicate)

        # Store the graph in the dictionary for the current timestep
        dynamic_graph[t] = directed_graph

    return dynamic_graph


# Example Usage

# Mock snapshots of DBpedia KG
snapshot_0 = {
    'http--dbpedia.org-resource-Category2000s_in_Turkey': {
        'http--www.w3.org-1999-02-22-rdf-syntax-ns#type': {'http--www.w3.org-2004-02-skos-core#Concept'},
        'http--www.w3.org-2004-02-skos-core#broader': {
            'http--dbpedia.org-resource-Category2000s_by_country',
            'http--dbpedia.org-resource-Category21st_century_in_Turkey'
        },
        'http--www.w3.org-2004-02-skos-core#prefLabel': {'"2000s in Turkey"@en'}
    },
    'http--dbpedia.org-resource-CategoryFL-Class_Australian_Antarctic_Territory_articles': {
        'http--www.w3.org-2004-02-skos-core#broader': {
            'http--dbpedia.org-resource-CategoryFL-Class_Australian_places_articles',
            'http--dbpedia.org-resource-CategoryFL-Class_articles',
            'http--dbpedia.org-resource-CategoryAustralian_Antarctic_Territory_articles_by_quality'
        },
        'http--www.w3.org-1999-02-22-rdf-syntax-ns#type': {'http--www.w3.org-2004-02-skos-core#Concept'},
        'http--www.w3.org-2004-02-skos-core#prefLabel': {'"FL-Class Australian Antarctic Territory articles"@en'}
    }
}

snapshot_1 = {
    'http--dbpedia.org-resource-Category21st_century_in_Turkey': {
        'http--www.w3.org-2004-02-skos-core#broader': {
            'http--dbpedia.org-resource-Category21st_century_by_country',
            'http--dbpedia.org-resource-Category2000s_in_Turkey'
        },
        'http--www.w3.org-1999-02-22-rdf-syntax-ns#type': {'http--www.w3.org-2004-02-skos-core#Concept'}
    }
}

# Combine snapshots into a list
snapshots = [snapshot_0, snapshot_1]

# Convert snapshots to graph dictionary
dynamic_graph = parse_triples_to_graph_dictionary(snapshots)

# Print the graph structure for each timestep
for t, graph in dynamic_graph.items():
    print(f"Timestep {t}:")
    print(f"Nodes: {list(graph.nodes())}")
    print(f"Edges: {list(graph.edges(data=True))}")


: 